# Lewisham · `application_type` (web: "Application type")

**Goal:** understand the data, so that a chat bot can answer any question about it
more easily and correctly. Lewisham also serves as a check on what we found for Brent.

Lewisham only (golden rule): every query carries the location-based Lewisham filter.
Data comes from the **live API** (the bot's only source), so counts change as the
Datahub is updated.

Workflow: add a step here → discuss the findings → only after agreement update
`semantic/lewisham/` and `knowledge/`.

In [1]:
from pathlib import Path

import httpx
import pandas as pd
from dotenv import dotenv_values

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
env = dotenv_values(PROJECT_ROOT / ".env")

BASE_URL = "https://planningdata.london.gov.uk/api-guest"
HEADERS = {"X-API-AllowRequest": env["DATAHUB_ALLOW_REQUEST"]}


def lewisham_filter() -> dict:
    # Location-based scope: borough is Lewisham, or borough is missing and Lewisham decided it.
    return {
        "bool": {
            "should": [
                {"term": {"borough.raw": "Lewisham"}},
                {"bool": {
                    "filter": [{"term": {"lpa_name.raw": "Lewisham"}}],
                    "must_not": [{"exists": {"field": "borough.raw"}}],
                }},
            ],
            "minimum_should_match": 1,
        }
    }


def search(body: dict) -> dict:
    response = httpx.post(f"{BASE_URL}/applications/_search", headers=HEADERS, json=body, timeout=60)
    response.raise_for_status()
    return response.json()

## Step 1 · How the Datahub records Prior Approval decisions

Two records the portal shows as "Granted Deemed Consent" are Prior Approval applications
(`knowledge/lewisham_portal_checks.md`). Here: every `decision` value for Lewisham's
Prior Approval applications, split by `status`, to see whether the Datahub already uses a
value for deemed cases.

In [2]:
body = {
    "size": 0,
    "track_total_hits": True,
    "query": {"bool": {"filter": [lewisham_filter(), {"term": {"application_type.raw": "Prior Approval"}}]}},
    "aggs": {
        "decision": {"terms": {"field": "decision.raw", "size": 100, "missing": "(none)"},
                     "aggs": {"status": {"terms": {"field": "status.raw", "size": 100, "missing": "(none)"}}}},
    },
}
data = search(body)
print("Lewisham Prior Approval applications:", data["hits"]["total"]["value"])
rows = [(d["key"], s["key"], s["doc_count"]) for d in data["aggregations"]["decision"]["buckets"]
        for s in d["status"]["buckets"]]
pa = pd.DataFrame(rows, columns=["decision", "status", "count"])
pa.pivot_table(index="decision", columns="status", values="count", aggfunc="sum", fill_value=0, margins=True, margins_name="total").sort_values("total", ascending=False)

Lewisham Prior Approval applications: 1598


status,(none),Allowed,Appeal Received,Application Under Consideration,Approved,Closed,Commenced,Completed,Dismissed,Lapsed,Not Required,Refused,Superseded,Withdrawn,total
decision,,,,,,,,,,,,,,,
total,21,8,1,10,57,2,3,112,29,25,686,584,13,47,1598
Not Required,0,0,0,0,0,0,1,13,0,0,686,0,1,0,701
Refused,0,8,1,0,0,0,0,1,29,0,0,584,0,2,625
Approved,0,0,0,0,57,0,2,98,0,24,0,0,11,0,192
Withdrawn,0,0,0,0,0,0,0,0,0,0,0,0,0,45,45
(none),21,0,0,10,0,0,0,0,0,1,0,0,1,0,33
Closed,0,0,0,0,0,2,0,0,0,0,0,0,0,0,2


In [3]:
# Any decision or status value anywhere in Lewisham that mentions "deemed" (case-insensitive).
body = {"size": 0, "query": lewisham_filter(), "aggs": {
    "decision": {"terms": {"field": "decision.raw", "size": 500}},
    "status": {"terms": {"field": "status.raw", "size": 500}},
}}
aggs = search(body)["aggregations"]
for field in ["decision", "status"]:
    hits = [(b["key"], b["doc_count"]) for b in aggs[field]["buckets"] if "deemed" in b["key"].lower()]
    print(field, "values with 'deemed':", hits or "none", "| distinct values:", len(aggs[field]["buckets"]))

decision values with 'deemed': none | distinct values: 9
status values with 'deemed': none | distinct values: 20


### Findings · Step 1

Live API, 2026-11-11, 1,598 Lewisham Prior Approval applications:

| `decision` | Count |
|---|---|
| Not Required | 701 |
| Refused | 625 |
| Approved | 192 |
| Withdrawn | 45 |
| (none) | 33 |
| Closed | 2 |

- **No "deemed" value anywhere.** No Lewisham `decision` (9 distinct values) or `status`
  (20) mentions "deemed". The Datahub has no value of its own for deemed cases.
- **Two "go ahead" outcomes exist:** `Approved` (192) and `Not Required` (701, nearly all
  with status Not Required too).
- **33 Prior Approvals have no decision:** 21 also have no status (one is
  DC/21/120320), 10 are Application Under Consideration, 1 Lapsed, 1
  Superseded.
- **DC/22/129646 is not counted here:** its `application_type` is All Other, although
  its `application_type_full` is "Prior Approval: Demolition of building".

## Step 2 · All `application_type` values

A `terms` aggregation on `application_type.raw` plus a `missing` aggregation, compared
with the schema's list. `shown_as` uses `repr()` so hidden characters are visible.

In [4]:
body = {
    "size": 0,
    "track_total_hits": True,
    "query": lewisham_filter(),
    "aggs": {
        "application_type": {"terms": {"field": "application_type.raw", "size": 200}},
        "no_application_type": {"missing": {"field": "application_type.raw"}},
    },
}
data = search(body)

total = data["hits"]["total"]["value"]
buckets = data["aggregations"]["application_type"]["buckets"]
type_counts = pd.DataFrame(
    [(b["key"], b["doc_count"]) for b in buckets] + [(None, data["aggregations"]["no_application_type"]["doc_count"])],
    columns=["application_type", "count"],
).sort_values("count", ascending=False, ignore_index=True)

print("Lewisham applications:", total, "| sum of counts:", type_counts["count"].sum())

Lewisham applications: 29021 | sum of counts: 29021


In [5]:
# From planninglondondatahub_public_technical_schemav2.1.xlsx, sheet "Field Values",
# index "applications", field "application_type" (extracted 2026-11-09).
SCHEMA_APPLICATION_TYPE = ["Householder", "Prior Approval", "All Other"]

type_counts["shown_as"] = type_counts["application_type"].map(repr)
type_counts["in_schema"] = type_counts["application_type"].isin(SCHEMA_APPLICATION_TYPE)
type_counts["share_%"] = (100 * type_counts["count"] / total).round(2)
type_counts

,application_type,count,shown_as,in_schema,share_%
0,All Other,20798,'All Other',True,71.67
1,Householder,6625,'Householder',True,22.83
2,Prior Approval,1598,'Prior Approval',True,5.51
3,NaN,0,nan,False,0.00


### Findings · Step 2

Live API, 2026-11-11, 29,021 Lewisham records:

| `application_type` | Count | Share | In schema |
|---|---|---|---|
| All Other | 20,798 | 71.67% | yes |
| Householder | 6,625 | 22.83% | yes |
| Prior Approval | 1,598 | 5.51% | yes |
| null | 0 | 0% | — |

- **Clean:** every record has a value, all 3 are schema values, no case or whitespace
  variants. Same as Brent.
- **Fewer Prior Approvals than Brent** (5.5% against 11.1%), more All Other (71.7% against
  63.2%). DC/22/129646 shows a Prior Approval can be typed All Other (Step 1), so the
  share may be understated; how often is not checked.

## Step 3 · Type check

Aggregations show every key as text, so the Python type of each `application_type` value
is checked on every record (paged with `search_after`, sorted by `id`).

In [6]:
rows, after = [], None
while True:
    body = {"size": 5000, "query": lewisham_filter(), "_source": ["id", "application_type"], "sort": [{"id": "asc"}]}
    if after:
        body["search_after"] = after
    page = search(body)["hits"]["hits"]
    if not page:
        break
    rows += [h["_source"] for h in page]
    after = page[-1]["sort"]

typed = pd.DataFrame(rows).reindex(columns=["id", "application_type"])
types = typed["application_type"].map(lambda v: "NoneType" if v is None or v != v else type(v).__name__)
print("Lewisham records:", len(typed))
print(types.value_counts().to_dict())

Lewisham records: 29021
{'str': 29021}


### Findings · Step 3

- **Every value is a string** (29,021); no nulls, no numbers. `application_type` matches
  the schema's text type.